# SmartCart – Data Cleaning

## Project: Global E-Commerce Orders & Returns Analysis

### Objective
To clean and prepare the five related datasets for accurate exploratory data analysis, business insights, and visualization.

### Datasets
1. Customers
2. Products
3. Orders
4. Order Items
5. Returns

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
project_path = Path.cwd().parent
raw_path = project_path / "Data" / "raw"

print("Project folder:", project_path)
print("Raw data folder:", raw_path)

Project folder: c:\Users\Aditya\Desktop\SmartCart-Analytics
Raw data folder: c:\Users\Aditya\Desktop\SmartCart-Analytics\Data\raw


In [3]:
customers = pd.read_csv(next(raw_path.rglob("customers.csv")))
products = pd.read_csv(next(raw_path.rglob("products.csv")))
orders = pd.read_csv(next(raw_path.rglob("SmartCart(Orders).csv")))
order_items = pd.read_csv(next(raw_path.rglob("order_items.csv")))
returns = pd.read_csv(next(raw_path.rglob("returns.csv")))

print("All five datasets loaded successfully!")

All five datasets loaded successfully!


In [4]:
customers_clean = customers.copy()
products_clean = products.copy()
orders_clean = orders.copy()
order_items_clean = order_items.copy()
returns_clean = returns.copy()

print("Working copies created successfully!")

Working copies created successfully!


In [5]:
datasets = {
    "Customers": customers_clean,
    "Products": products_clean,
    "Orders": orders_clean,
    "Order Items": order_items_clean,
    "Returns": returns_clean
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

Customers: (3000, 12)
Products: (345, 10)
Orders: (14000, 23)
Order Items: (29671, 11)
Returns: (2433, 15)


In [6]:
orders_clean = orders_clean.drop(
    columns=["Unnamed: 20", "Unnamed: 21", "Unnamed: 22"]
)

print("Updated Orders shape:", orders_clean.shape)

Updated Orders shape: (14000, 20)


In [7]:
for name, df in datasets.items():
    print(f"\n{name}")
    print(df.isnull().sum()[df.isnull().sum() > 0])


Customers
first_order_date    34
dtype: int64

Products
Series([], dtype: int64)

Orders
delivery_days     1246
Unnamed: 20      13999
Unnamed: 21      14000
Unnamed: 22      13999
dtype: int64

Order Items
Series([], dtype: int64)

Returns
Series([], dtype: int64)


In [8]:
datasets["Orders"] = orders_clean

In [9]:
datasets["Orders"] = orders_clean

print("Orders entry updated:", datasets["Orders"].shape)

Orders entry updated: (14000, 20)


In [10]:
for name, df in datasets.items():
    print(f"\n{name}")
    print(df.isnull().sum()[df.isnull().sum() > 0])


Customers
first_order_date    34
dtype: int64

Products
Series([], dtype: int64)

Orders
delivery_days    1246
dtype: int64

Order Items
Series([], dtype: int64)

Returns
Series([], dtype: int64)


### Missing Value Treatment

- Customers: Preserved 34 missing first_order_date values because these customers have zero lifetime orders.
- Orders: Preserved 1,246 missing delivery_days values because these orders are cancelled or still processing.

Decision: No missing values were filled or deleted because they represent valid business conditions.

In [11]:
for name, df in datasets.items():
    print(f"{name}: {df.duplicated().sum()} duplicate rows")

Customers: 0 duplicate rows
Products: 0 duplicate rows
Orders: 0 duplicate rows
Order Items: 0 duplicate rows
Returns: 0 duplicate rows


In [12]:
primary_keys = {
    "Customers": ("customer_id", customers_clean),
    "Products": ("product_id", products_clean),
    "Orders": ("order_id", orders_clean),
    "Order Items": ("order_item_id", order_items_clean),
    "Returns": ("return_id", returns_clean)
}

for name, (column, df) in primary_keys.items():
    duplicates = df[column].duplicated().sum()
    print(f"{name} - Duplicate {column}: {duplicates}")

Customers - Duplicate customer_id: 0
Products - Duplicate product_id: 0
Orders - Duplicate order_id: 0
Order Items - Duplicate order_item_id: 0
Returns - Duplicate return_id: 0


In [13]:
for name, df in datasets.items():
    print(f"\n--- {name} ---")
    print(df.dtypes)


--- Customers ---
customer_id                str
customer_name              str
country                    str
region                     str
currency_code              str
signup_date                str
acquisition_channel        str
preferred_device           str
newsletter_opt_in          str
first_order_date           str
lifetime_orders          int64
lifetime_spend_usd     float64
dtype: object

--- Products ---
product_id            str
product_name          str
category              str
subcategory           str
brand                 str
list_price_usd    float64
unit_cost_usd     float64
weight_kg         float64
launch_date           str
is_active             str
dtype: object

--- Orders ---
order_id                  str
customer_id               str
order_date                str
order_month               str
sales_channel             str
device_type               str
payment_method            str
shipping_country          str
shipping_region           str
currency_code    

In [14]:
for name, df in datasets.items():
    print(f"\n{name}")
    print("String columns:", len(df.select_dtypes(include=["str"]).columns))
    print("Numeric columns:", len(df.select_dtypes(include=["number"]).columns))
    print("Datetime columns:", len(df.select_dtypes(include=["datetime"]).columns))


Customers
String columns: 10
Numeric columns: 2
Datetime columns: 0

Products
String columns: 7
Numeric columns: 3
Datetime columns: 0

Orders
String columns: 11
Numeric columns: 9
Datetime columns: 0

Order Items
String columns: 4
Numeric columns: 7
Datetime columns: 0

Returns
String columns: 11
Numeric columns: 4
Datetime columns: 0


In [15]:
customers_clean["signup_date"] = pd.to_datetime(
    customers_clean["signup_date"],
    format="%d-%m-%Y"
)

customers_clean["first_order_date"] = pd.to_datetime(
    customers_clean["first_order_date"],
    format="%d-%m-%Y"
)

print(customers_clean[["signup_date", "first_order_date"]].dtypes)

signup_date         datetime64[us]
first_order_date    datetime64[us]
dtype: object


In [16]:
products_clean["launch_date"] = pd.to_datetime(
    products_clean["launch_date"],
    format="%d-%m-%Y"
)

print(products_clean["launch_date"].dtype)

datetime64[us]


In [17]:
orders_clean["order_date_corrected"] = pd.NaT

slash_mask = orders_clean["order_date"].str.contains("/", na=False)
hyphen_mask = orders_clean["order_date"].str.contains("-", na=False)

orders_clean.loc[slash_mask, "order_date_corrected"] = pd.to_datetime(
    orders_clean.loc[slash_mask, "order_date"],
    format="%d/%m/%Y"
)

orders_clean.loc[hyphen_mask, "order_date_corrected"] = pd.to_datetime(
    orders_clean.loc[hyphen_mask, "order_date"],
    format="%d-%m-%Y"
)

orders_clean["order_month"] = pd.to_datetime(
    orders_clean["order_month"],
    format="%Y-%m"
)

print(orders_clean[["order_date_corrected", "order_month"]].dtypes)
print("Missing corrected dates:", orders_clean["order_date_corrected"].isna().sum())

order_date_corrected    datetime64[ns]
order_month             datetime64[us]
dtype: object
Missing corrected dates: 0


In [18]:
returns_clean["return_date"] = pd.to_datetime(
    returns_clean["return_date"],
    format="%d-%m-%Y"
)

returns_clean["return_month"] = pd.to_datetime(
    returns_clean["return_month"],
    format="%Y-%m"
)

print(returns_clean[["return_date", "return_month"]].dtypes)

return_date     datetime64[us]
return_month    datetime64[us]
dtype: object


In [19]:
for name, df in datasets.items():
    print(f"\n{name}")
    print(df.select_dtypes(include=["datetime"]).columns.tolist())


Customers
['signup_date', 'first_order_date']

Products
['launch_date']

Orders
['order_month', 'order_date_corrected']

Order Items
[]

Returns
['return_date', 'return_month']


In [20]:
datasets = {
    "Customers": customers_clean,
    "Products": products_clean,
    "Orders": orders_clean,
    "Order Items": order_items_clean,
    "Returns": returns_clean
}

In [21]:
for name, df in datasets.items():
    numeric_columns = df.select_dtypes(include=["number"]).columns.tolist()
    print(f"\n{name}:")
    print(numeric_columns)


Customers:
['lifetime_orders', 'lifetime_spend_usd']

Products:
['list_price_usd', 'unit_cost_usd', 'weight_kg']

Orders:
['fx_rate_to_usd', 'items_count', 'gross_amount_local', 'discount_local', 'shipping_fee_local', 'tax_local', 'order_total_local', 'order_total_usd', 'delivery_days']

Order Items:
['quantity', 'unit_price_local', 'line_discount_local', 'line_total_local', 'line_total_usd', 'unit_cost_usd', 'line_profit_usd']

Returns:
['days_to_return', 'quantity_returned', 'refund_amount_local', 'refund_amount_usd']


In [22]:
for name, df in datasets.items():
    numeric_columns = df.select_dtypes(include=["number"]).columns

    print(f"\n--- {name} ---")

    for column in numeric_columns:
        negative_count = (df[column] < 0).sum()

        if negative_count > 0:
            print(f"{column}: {negative_count} negative values")


--- Customers ---

--- Products ---

--- Orders ---

--- Order Items ---
line_profit_usd: 484 negative values

--- Returns ---


### Negative Value Validation

- Checked all numeric columns across the five datasets.
- Found 484 negative values in order_items.line_profit_usd.
- These values represent transactions with negative profit (business losses).
- No negative values were found in other numeric columns.

Decision: Preserved all negative profit values because they are meaningful for business analysis.

In [23]:
processed_path = project_path / "Data" / "processed"

processed_path.mkdir(parents=True, exist_ok=True)

print("Processed data folder:", processed_path)
print("Folder ready!")

Processed data folder: c:\Users\Aditya\Desktop\SmartCart-Analytics\Data\processed
Folder ready!


In [24]:
customers_clean.to_csv(processed_path / "customers_clean.csv", index=False)
products_clean.to_csv(processed_path / "products_clean.csv", index=False)
orders_clean.to_csv(processed_path / "orders_clean.csv", index=False)
order_items_clean.to_csv(processed_path / "order_items_clean.csv", index=False)
returns_clean.to_csv(processed_path / "returns_clean.csv", index=False)

print("All five cleaned datasets saved successfully!")

All five cleaned datasets saved successfully!


In [25]:
for file in processed_path.iterdir():
    print(file.name, "-", round(file.stat().st_size / 1024, 2), "KB")

customers_clean.csv - 300.83 KB
orders_clean.csv - 2335.22 KB
order_items_clean.csv - 2184.76 KB
products_clean.csv - 31.03 KB
returns_clean.csv - 304.53 KB


In [26]:
saved_datasets = {
    "Customers": pd.read_csv(processed_path / "customers_clean.csv"),
    "Products": pd.read_csv(processed_path / "products_clean.csv"),
    "Orders": pd.read_csv(processed_path / "orders_clean.csv"),
    "Order Items": pd.read_csv(processed_path / "order_items_clean.csv"),
    "Returns": pd.read_csv(processed_path / "returns_clean.csv")
}

for name, df in saved_datasets.items():
    print(f"{name}: {df.shape}")

Customers: (3000, 12)
Products: (345, 10)
Orders: (14000, 21)
Order Items: (29671, 11)
Returns: (2433, 15)


# Final Data Cleaning Summary – SmartCart

## 1. Dataset Preparation
- Loaded all five original CSV datasets.
- Created separate working copies.
- Removed three unnecessary columns from Orders.
- Preserved original raw datasets.

## 2. Missing Values
- Customers: 34 missing first_order_date values preserved.
- Orders: 1,246 missing delivery_days values preserved.
- No unnecessary filling or deletion performed.

## 3. Duplicate Validation
- No duplicate rows found.
- No duplicate primary keys found across all five datasets.

## 4. Data Type Corrections
- Converted all date columns into datetime format.
- Preserved original order_date and created order_date_corrected.
- Numeric columns were already correctly formatted.

## 5. Numeric Validation
- Found 484 negative values in line_profit_usd.
- Preserved them because they represent genuine business losses.

## 6. Exported Cleaned Datasets
- customers_clean.csv
- products_clean.csv
- orders_clean.csv
- order_items_clean.csv
- returns_clean.csv

## 7. Final Dataset Dimensions
- Customers: 3,000 rows
- Products: 345 rows
- Orders: 14,000 rows
- Order Items: 29,671 rows
- Returns: 2,433 rows

## Conclusion
All five datasets have been cleaned, validated, and exported into the Data/processed folder. The cleaned datasets are ready for exploratory data analysis and visualization.